# Proyecto · Clase 7 — Redes convolucionales: ver como ve una cámara

Nuestra red mira **cada píxel por separado**: el píxel 300 tiene sus pesos y el 301 los suyos. No sabe que
están uno al lado del otro. Eso tiene una consecuencia fea, que vamos a ver con tus propios pesos.

In [ ]:
import os, urllib.request
for archivo in ["digitos.py", "dibujar.py"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/datahuba/curso-redes-neuronales/main/proyecto/" + archivo, archivo)

import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from digitos import cargar_mnist, mostrar, ok, predecir_con_pesos

X_train, y_train, X_test, y_test = cargar_mnist()
X_train, X_test = X_train / 255, X_test / 255         # ¡esta vez NO aplanamos! nos importa la forma 28x28

## 1. El experimento: mover el dibujo
Movemos todas las imágenes del examen 3 píxeles a la derecha y 3 hacia abajo. Para una persona es el mismo número.

In [ ]:
movidas = np.roll(X_test, (3, 3), axis=(1, 2))
mostrar(np.concatenate([X_test[:5], movidas[:5]]), titulos=["original"] * 5 + ["movida"] * 5, columnas=5)

if os.path.exists("pesos.npz"):
    pesos = dict(np.load("pesos.npz"))
    for nombre, imgs in [("originales", X_test), ("movidas", movidas)]:
        acc = (predecir_con_pesos(pesos, imgs.reshape(-1, 784)).argmax(axis=1) == y_test).mean()
        print(f"Tu red densa con imágenes {nombre}: {acc:.1%}")
else:
    print("No encuentro pesos.npz: ejecuta antes el notebook de la Clase 5 o 6.")

😱 Aprendió **«en qué lugar»** suele haber tinta, no **«qué forma»** tiene el número.

## 2. La convolución: una lupa que recorre la imagen
Un **filtro** es una tablita de 3 × 3 números. Lo ponemos sobre cada rincón de la imagen, multiplicamos casilla
por casilla y sumamos: un número por posición. **El mismo filtro en todos lados** → si detecta una curva arriba,
también la detecta abajo.

✏️ **Ejercicio 1.** Escribe la convolución con dos ciclos. La salida mide 26 × 26 (el filtro no cabe en el borde).

In [ ]:
def convolucion(img, filtro):
    salida = np.zeros((26, 26))
    for i in range(26):
        for j in range(26):
            salida[i, j] = ...      # recorte 3x3 en (i, j) por el filtro, todo sumado
    return salida

filtros = {
    "bordes verticales": np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]]),
    "bordes horizontales": np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]]),
    "desenfoque": np.ones((3, 3)) / 9,
}
fig, ejes = plt.subplots(1, 4, figsize=(12, 3))
ejes[0].imshow(X_train[0], cmap="gray"); ejes[0].set_title("original")
for ax, (nombre, f) in zip(ejes[1:], filtros.items()):
    ax.imshow(convolucion(X_train[0], f), cmap="gray"); ax.set_title(nombre)
for ax in ejes:
    ax.axis("off")
plt.show()

In [ ]:
ok(np.isclose(convolucion(np.ones((28, 28)), np.ones((3, 3)))[5, 5], 9), "la convolución suma 9 productos")

**La gran idea:** en una red convolucional **no elegimos los filtros a mano**. Sus 9 números son pesos, y la red
los aprende con descenso por gradiente, igual que todo lo demás.

## 3. La versión rápida
Los dos ciclos son lentos para 60 000 imágenes. Truco: sacar **todos los recortes 3 × 3** de una vez (cada
recorte = una fila de 9) y multiplicar por los filtros con un solo `@`. Mismo resultado, mil veces más rápido.

In [ ]:
def recortes(imgs):
    """(n, 28, 28) → (n, 26, 26, 9): los 9 píxeles de cada recorte 3x3."""
    return sliding_window_view(imgs, (3, 3), axis=(1, 2)).reshape(len(imgs), 26, 26, 9)

F = np.stack([f.ravel() for f in filtros.values()], axis=1)              # 9 x 3: un filtro por columna
rapido = recortes(X_train[:1]) @ F
ok(np.allclose(rapido[0, :, :, 0], convolucion(X_train[0], filtros["bordes verticales"])), "la versión rápida da lo mismo")

## 4. Nuestra red convolucional
```
imagen 28x28 → 8 filtros 3x3 + ReLU → 26x26x8 → max-pooling 2x2 → 13x13x8 = 1352 → 10 salidas (softmax)
```
**Max-pooling:** de cada cuadrito de 2 × 2 nos quedamos con el valor más alto. Reduce el tamaño y hace que mover
el dibujo un poco importe menos.

✏️ **Ejercicio 2.** Completa el pase hacia adelante. (La retropropagación te la damos: es la misma receta de la
Clase 5, con dos detalles nuevos explicados en los comentarios.)

In [ ]:
def relu(Z):
    return np.maximum(0, Z)


def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)


def max_pool(A):
    """(n, 26, 26, c) → (n, 13, 13, c): el máximo de cada cuadrito 2x2."""
    n, _, _, c = A.shape
    return A.reshape(n, 13, 2, 13, 2, c).max(axis=(2, 4))


def adelante(red, imgs):
    R = recortes(imgs)                                    # (n, 26, 26, 9)
    Z = ...                                           # convolución: recortes @ filtros + sesgo de cada filtro
    A = ...                                           # ReLU
    M = max_pool(A)                                       # (n, 13, 13, 8)
    plano = M.reshape(len(imgs), -1)                      # (n, 1352)
    P = ...                                           # capa densa + softmax (como en la Clase 4)
    return R, Z, A, M, plano, P


def gradientes(red, imgs, Y):
    R, Z, A, M, plano, P = adelante(red, imgs)
    n = len(imgs)
    dZ2 = (P - Y) / n                                     # error de la salida (igual que siempre)
    g = {"W": plano.T @ dZ2, "b": dZ2.sum(axis=0)}
    dM = (dZ2 @ red["W"].T).reshape(M.shape)              # culpa de cada valor del pooling
    # Detalle nuevo 1: en el max-pooling solo el ganador de cada cuadrito recibe la culpa.
    ganador = A == np.repeat(np.repeat(M, 2, axis=1), 2, axis=2)
    dA = ganador * np.repeat(np.repeat(dM, 2, axis=1), 2, axis=2)
    dZ = dA * (Z > 0)                                     # ReLU, como en la Clase 5
    # Detalle nuevo 2: el filtro se usó en las 676 posiciones → sumamos su pendiente de todas ellas.
    g["F"] = R.reshape(-1, 9).T @ dZ.reshape(-1, red["F"].shape[1])
    g["bf"] = dZ.sum(axis=(0, 1, 2))
    return g


def crear_cnn(n_filtros=8, semilla=0):
    rng = np.random.default_rng(semilla)
    return {"F": rng.normal(0, np.sqrt(2 / 9), (9, n_filtros)), "bf": np.zeros(n_filtros),
            "W": rng.normal(0, np.sqrt(2 / 1352), (1352, 10)), "b": np.zeros(10)}

In [ ]:
ok(adelante(crear_cnn(), X_train[:4])[-1].shape == (4, 10), "la red convolucional da 10 probabilidades por imagen")

## 5. Entrenar (con momentum, de la Clase 6)
Usamos 20 000 imágenes y 3 épocas para que tarde ~1 minuto.

In [ ]:
def precision(red, imgs, y, lote=1000):
    return np.mean(np.concatenate([adelante(red, imgs[i:i + lote])[-1].argmax(axis=1) == y[i:i + lote]
                                   for i in range(0, len(imgs), lote)]))


def entrenar(red, imgs, y, epocas=3, eta=0.05, beta=0.9, lote=64, semilla=0):
    rng = np.random.default_rng(semilla)
    Y, v = np.eye(10)[y], {k: 0 for k in red}
    for epoca in range(epocas):
        orden = rng.permutation(len(imgs))
        for i in range(0, len(imgs), lote):
            idx = orden[i:i + lote]
            g = gradientes(red, imgs[idx], Y[idx])
            for k in red:
                v[k] = beta * v[k] - eta * g[k]
                red[k] += v[k]
        print(f"época {epoca + 1}: precisión en el examen {precision(red, X_test, y_test):.2%}")
    return red

cnn = entrenar(crear_cnn(), X_train[:20000], y_train[:20000])

In [ ]:
acc, acc_mov = precision(cnn, X_test, y_test), precision(cnn, movidas, y_test)
print(f"Red convolucional · originales: {acc:.1%}   movidas: {acc_mov:.1%}")
ok(acc > 0.95, "la red convolucional supera el 95 %")

Mejora un poco con las movidas, pero **no lo resuelve**: los filtros detectan formas en cualquier lugar, pero la
capa densa del final todavía aprendió *dónde* suelen estar.

## 6. El truco de los profesionales: entrenar con imágenes movidas (*aumento de datos*)
Si queremos que no le importe la posición, **le enseñamos ejemplos movidos**. No hace falta conseguir imágenes
nuevas: las fabricamos moviendo las que ya tenemos.

✏️ **Ejercicio 3.** Mueve cada imagen una cantidad al azar entre −3 y 3 píxeles en cada eje.
Pista: `np.roll(img, (dy, dx), axis=(0, 1))`.

In [ ]:
def mover_al_azar(imgs, maximo=3, semilla=1):
    rng = np.random.default_rng(semilla)
    salida = np.empty_like(imgs)
    for i, img in enumerate(imgs):
        dy, dx = rng.integers(-maximo, maximo + 1, size=2)
        salida[i] = ...
    return salida

aumentadas = mover_al_azar(X_train[:20000])
mostrar(aumentadas[:10], titulos=y_train[:10])
cnn_mov = entrenar(crear_cnn(), aumentadas, y_train[:20000])

In [ ]:
acc2, acc2_mov = precision(cnn_mov, X_test, y_test), precision(cnn_mov, movidas, y_test)
print(f"Convolucional entrenada con movidas · originales: {acc2:.1%}   movidas: {acc2_mov:.1%}")
ok(acc2_mov > acc_mov + 0.3, "con aumento de datos la red resiste los movimientos")

**Para comentar:** perdió un poquito con las originales y ganó muchísimo con las movidas. ¿Qué otros
«aumentos» se te ocurren para dígitos escritos a mano? (rotar un poco, hacer el trazo más grueso…)

## 7. ¿Qué filtros aprendió?
Nadie se los dijo. Compáralos con los de la sección 2.

In [ ]:
fig, ejes = plt.subplots(2, 8, figsize=(14, 4))
for k in range(8):
    ejes[0, k].imshow(cnn["F"][:, k].reshape(3, 3), cmap="bwr"); ejes[0, k].set_title(f"filtro {k}")
    ejes[1, k].imshow(adelante(cnn, X_test[:1])[2][0, :, :, k], cmap="gray")
for ax in ejes.flat:
    ax.axis("off")
plt.show()

## 8. Experimentos 🔁
- Más filtros (`crear_cnn(16)`) o más imágenes: ¿mejora? ¿Cuánto tarda?
- ¿Y si entrenas la red **densa** de la Clase 6 con imágenes movidas?
- Las redes reales apilan **varias** capas conv + pooling: cada capa ve un pedazo más grande de la imagen.

## 9. Cierre del curso: lo mismo, en la vida real
Todo lo que escribiste a mano (capas, softmax, retropropagación, momentum) es lo que hacen por dentro
**PyTorch** y **TensorFlow**. Así se ve tu red convolucional en PyTorch (no hace falta ejecutarlo):
```python
red = nn.Sequential(
    nn.Conv2d(1, 8, 3), nn.ReLU(), nn.MaxPool2d(2),   # tus filtros + ReLU + pooling
    nn.Flatten(), nn.Linear(1352, 10))                # tu capa densa
optimizador = torch.optim.SGD(red.parameters(), lr=0.05, momentum=0.9)
for x, y in datos:
    error = nn.functional.cross_entropy(red(x), y)    # tu softmax + entropía cruzada
    optimizador.zero_grad(); error.backward()          # tu retropropagación, automática
    optimizador.step()                                 # tu momentum
```
La diferencia: tú **sabes qué pasa dentro de cada línea**. 🎓

| Clase | Pieza que agregaste | Precisión |
|---|---|---|
| 1 | Plantillas promedio | ~82 % |
| 2 | Una neurona (perceptrón) | 0 vs 1: ~99.8 % |
| 3 | Sigmoide + descenso por gradiente | 3 vs 5: ~96 % |
| 4 | 10 neuronas + softmax | ~92 % |
| 5 | Capa oculta + retropropagación | ~97 % |
| 6 | Momentum | ~97.6 % |
| 7 | Convolución + pooling + aumento de datos | ~97 % con solo 20 000 imágenes; con aumento resiste movimientos (81 % vs 17 %) |